In [3]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x10c44f750>)

In [4]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [5]:
print("Length of text:" , len(text))

Length of text: 1115394


In [13]:
print(text[0:450])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, go


In [18]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### Tokenizer

In [28]:
#BPE, WordPiece, Unigram, SentencePiece are other alternatives

In [ ]:
#Creating a mapping by assigning an integer to each charachter in the corpus, we do this simply indexing the charachters after sorting them.

#Mapping from char to integer
stoi = {ch:i for i, ch in enumerate(chars)}
#Mapping from integer to string
itos = {i:ch for i, ch in enumerate(chars)}

In [24]:
encoder = lambda s: [stoi[c] for c in s]
decoder = lambda s: ''.join([itos[i] for i in s])

In [26]:
sentence = "hi how are you?"
print(encoder(sentence))
print(decoder(encoder(sentence)))

[46, 47, 1, 46, 53, 61, 1, 39, 56, 43, 1, 63, 53, 59, 12]
hi how are you?


In [30]:
pip install torch

  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 1.1 MB/s  0:01:23m0:00:0100:020m
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.29.0-py3-none-any.whl (39 kB)
  Attempting uninstall: setuptools━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [sympy]
    Found existing installation: setuptools 82.0.1━━━━━━━━━━━━ 1/7 [sympy]
    Uninstalling setuptools-82.0.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [sympy]
      Successfully uninstalled setuptools-82.0.1━━━━━━━━━━━━━━━━━━ 2/7 [setuptools]
   ━

In [32]:
import torch

/opt/miniconda3/envs/venv_gpt_scratch/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [43]:
data = torch.tensor(encoder(text), dtype = torch.long)

In [54]:
data[0:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

### Train Test Split

In [61]:
n = int(0.9*len(data))
train = data[0:n]
val = data[n:]
print(n, len(train), len(val), len(train)+len(val))

1003854 1003854 111540 1115394


In [68]:
train[0:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [ ]:
block_size = 8
# A block size of 8, essentially means that we are trying to have 8 predictions in the future, 
# hence for 8 charactwrs, we would have 8+1 charachters considered
train[0:block_size + 1]
#To fully understand, see the code below
x = train[0:block_size]
y = train[1:block_size+1] #Note how y skips the first charachter and starts from the second charachter, 
    #this is because we are trying to predict the next charachter in the sequence

for i in range(0, block_size):
    context = x[:i+1]
    target = y[i] 
    print(f"When contect is {context} the target: {target}")

When contect is tensor([18]) the target: 47
When contect is tensor([18, 47]) the target: 56
When contect is tensor([18, 47, 56]) the target: 57
When contect is tensor([18, 47, 56, 57]) the target: 58
When contect is tensor([18, 47, 56, 57, 58]) the target: 1
When contect is tensor([18, 47, 56, 57, 58,  1]) the target: 15
When contect is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
When contect is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
torch.manual_seed(1337)
batch_size = 4 #Number of sequences to process in parallel
block_size = 8 #Max context length essentially 

def get_batch(data):
    data = data
    ix = torch.randint((len(data) - block_size), (batch_size,)) 
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [74]:
xb, yb = get_batch(train)
print("Inputs: ", xb)
print("Targets: ", yb)

Inputs:  tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Targets:  tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [76]:
for i in range(batch_size):
    print(f"Example {i+1}:")
    context = xb[i]
    target = yb[i]
    for t in range(block_size):
        print(f"When context is {context[:t+1]} the target is {target[t]}")

Example 1:
When context is tensor([24]) the target is 43
When context is tensor([24, 43]) the target is 58
When context is tensor([24, 43, 58]) the target is 5
When context is tensor([24, 43, 58,  5]) the target is 57
When context is tensor([24, 43, 58,  5, 57]) the target is 1
When context is tensor([24, 43, 58,  5, 57,  1]) the target is 46
When context is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
When context is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
Example 2:
When context is tensor([44]) the target is 53
When context is tensor([44, 53]) the target is 56
When context is tensor([44, 53, 56]) the target is 1
When context is tensor([44, 53, 56,  1]) the target is 58
When context is tensor([44, 53, 56,  1, 58]) the target is 46
When context is tensor([44, 53, 56,  1, 58, 46]) the target is 39
When context is tensor([44, 53, 56,  1, 58, 46, 39]) the target is 58
When context is tensor([44, 53, 56,  1, 58, 46, 39, 58]) the target is 1
Example 3:
When contex

### Bigram Model

In [77]:
import torch.nn
from torch.nn import functional as F

In [113]:
class BigramLanguageModel(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = torch.nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx) #Batch, Time and Channel 
        if targets is None:
            loss = None
            
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        #idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits, loss = self(idx)
            #focus only on the last time step
            logits = logits[:, -1, :] # becomes (B,C) What is B,T,C here? Answer : B is the batch size, T is the time step and C is the number of channels or vocab size. 
            probs = F.softmax(logits, dim = -1) #B,C Here what we are doing is that we are applying the softmax function to the logits to get the probabilities of the next token, and we are doing this for each example in the batch.
            idx_next = torch.multinomial(probs, num_samples = 1) #B,1. Here what are we doing is that we are sampling from the probability distribution of the next token, and we are doing this for each example in the batch.
            #append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim = 1) #B,T+1 Here what we are doing is that we are concatenating the new index to the existing sequence, hence increasing the time step by 1.
        return idx



m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

#idx = torch.zeroes((1,1), dtype = torch.long)
print(decoder(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 100)[0].tolist())) 


torch.Size([256, 65])
tensor(4.7850, grad_fn=<NllLossBackward0>)

khYOeCxhZ'RQ p;-L?.lshrsgtcojre3Iyx&oALGegt qvaScoTCsc!cPpgJlVOeFlBsbaKmiE&rFTS!FTPN'RYRqtaOeKpGL&d;


In [116]:
#create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-2)

In [117]:
batch_size = 32
for steps in range(10000):

    #sample a batch of data
    xb, yb = get_batch(train)

    #evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none =True) #Here the function zero_grad is used to reset the gradients of the model parameters before we perform backpropagation, and the argument set_to_none = True is used to set the gradients to None instead of zero, which can be more memory efficient in some cases.
    loss.backward()
    optimizer.step() 
print(loss.item())

2.4437711238861084


In [122]:
print(decoder(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 100)[0].tolist())) 


As wn tede fotofe ouproweofuamouthan herlll or o bl rere

The ler tlld ket idit t;

le, crisod se po
